# 🧠 Week 2: LLM APIs — Function Calling, Streaming & Prompt Engineering

**Goal:** Master LLM API usage — the **brain** of every AI agent.

**Why this matters:** An agent's intelligence comes entirely from how it interacts with LLMs. 
This week you'll learn to:
- Call OpenAI and Anthropic APIs correctly
- Use function calling (how agents invoke tools)
- Stream responses token-by-token
- Engineer prompts that control agent behavior
- Get structured outputs reliably
- Track tokens and manage costs

---

| Section | Topic | Agent Relevance |
|---------|-------|-----------------|
| 1 | OpenAI API Deep Dive | The most common agent backend |
| 2 | Function Calling & Tool Use | How LLMs invoke agent tools |
| 3 | Streaming Responses | Real-time output to users |
| 4 | Anthropic Claude API | Alternative/better reasoning model |
| 5 | Prompt Engineering for Agents | Controlling agent behavior |
| 6 | Structured Outputs | Reliable JSON from LLMs |
| 7 | Token Management & Cost | Production cost control |
| 8 | Capstone: LLM Router | Smart model selection |

In [ ]:
# Install dependencies
!pip install openai anthropic tiktoken -q

---
# Section 1: OpenAI API Deep Dive

## The Most Important API for Agent Builders

OpenAI's Chat Completions API is the most widely used LLM API. Understanding it deeply 
is essential because:
- Most agent frameworks default to OpenAI
- The function calling format has become the de facto standard
- Other providers (Azure, local models via Ollama/vLLM) use the same API format

### API Architecture
```
Your Agent Code
      │
      ▼
┌──────────────────┐     ┌──────────────────┐
│  OpenAI Python   │────►│  api.openai.com  │
│  Client Library  │◄────│  (/v1/chat/      │
│                  │     │  completions)     │
└──────────────────┘     └──────────────────┘
                                │
                                ▼
                         ┌──────────────┐
                         │  GPT-4o /    │
                         │  GPT-4o-mini │
                         │  o3 / o4-mini│
                         └──────────────┘
```

### 1.1 Basic Chat Completion

In [ ]:
import openai
import os
import json

# Initialize the client
# The API key is loaded from OPENAI_API_KEY environment variable
client = openai.OpenAI()

# --- Basic Chat Completion ---
response = client.chat.completions.create(
    model="gpt-4o-mini",  # Cost-effective for learning
    messages=[
        {
            "role": "system",
            "content": "You are a helpful AI assistant. Be concise."
        },
        {
            "role": "user", 
            "content": "What is an AI agent in 2 sentences?"
        }
    ],
    temperature=0.7,   # 0 = deterministic, 1 = creative, 2 = very random
    max_tokens=150,     # Maximum output tokens
)

# --- Inspect the response object ---
print("=== Response Object ===")
print(f"Model: {response.model}")
print(f"Content: {response.choices[0].message.content}")
print(f"Finish reason: {response.choices[0].finish_reason}")
print(f"\n=== Token Usage ===")
print(f"Prompt tokens: {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")

### 1.2 Message Roles — The Conversation Structure

Understanding roles is crucial for building agents:

In [ ]:
# The 4 message roles in OpenAI's API
messages = [
    # SYSTEM: Sets the agent's persona, rules, and constraints
    # This is where you define HOW the agent should behave
    {
        "role": "system",
        "content": (
            "You are a data analyst agent. You help users analyze data. "
            "Always explain your reasoning step by step. "
            "If you need to run a calculation, use the calculator tool. "
            "Never make up data — always use tools to verify."
        )
    },
    
    # USER: Input from the human (or from the environment)
    {
        "role": "user",
        "content": "What was our revenue last quarter?"
    },
    
    # ASSISTANT: Previous LLM responses (for conversation context)
    {
        "role": "assistant",
        "content": "I'll look up the revenue data for you. Let me query the database."
    },
    
    # TOOL: Results from tool execution (we'll cover this in Section 2)
    # {
    #     "role": "tool",
    #     "tool_call_id": "call_abc123",
    #     "content": '{"revenue": 4200000, "quarter": "Q3 2025"}'
    # },
    
    # Another user follow-up
    {
        "role": "user",
        "content": "How does that compare to Q2?"
    },
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=0,  # Deterministic for data tasks
    max_tokens=200,
)

print(f"Assistant: {response.choices[0].message.content}")
print(f"\n💡 The LLM sees the ENTIRE message history — this IS the agent's 'memory'")

### 1.3 Key Parameters & Their Agent Implications

| Parameter | Values | Agent Use Case |
|-----------|--------|----------------|
| `model` | `gpt-4o`, `gpt-4o-mini`, `o3`, `o4-mini` | Quality vs cost tradeoff |
| `temperature` | 0.0 - 2.0 | 0 for tools/data, 0.7 for creative tasks |
| `max_tokens` | 1 - 16384 | Control output length and cost |
| `top_p` | 0.0 - 1.0 | Alternative to temperature (use one, not both) |
| `frequency_penalty` | -2.0 - 2.0 | Reduce repetition in long outputs |
| `presence_penalty` | -2.0 - 2.0 | Encourage topic diversity |
| `stop` | list of strings | Stop generation at specific tokens |
| `seed` | integer | Reproducible outputs (best effort) |
| `response_format` | `json_object`, `json_schema` | Force structured output |

In [ ]:
# --- Temperature comparison ---
# Low temperature = deterministic (good for agent decisions)
# High temperature = creative (good for brainstorming)

prompt = "Name 3 programming languages for AI development."

for temp in [0.0, 0.7, 1.5]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        max_tokens=80,
    )
    print(f"Temperature {temp}: {response.choices[0].message.content[:100]}...")
    print()

print("💡 For agent tool-calling decisions, ALWAYS use temperature=0")
print("   For creative tasks, use 0.5-0.9")

---
# Section 2: Function Calling & Tool Use

## The Core of Agent Intelligence

**Function calling** is the mechanism that turns a chatbot into an agent.
Instead of just generating text, the LLM can request to call specific functions.

```
Without function calling:
  User: "What's the weather in Paris?"
  LLM:  "I don't have real-time data, but typically..." ← Hallucinating!

With function calling:
  User: "What's the weather in Paris?"
  LLM:  → Calls get_weather(city="Paris")  ← Uses a tool!
  Tool: Returns {"temp": "22°C", "condition": "Sunny"}
  LLM:  "The weather in Paris is 22°C and Sunny." ← Accurate!
```

### How Function Calling Works

```
Step 1: You define tools (JSON schemas)
Step 2: Send tools + messages to LLM
Step 3: LLM decides to call a tool (returns tool_calls)
Step 4: You execute the tool
Step 5: Send tool result back to LLM
Step 6: LLM generates final response using tool result
```

### 2.1 Defining Tools for the LLM

In [ ]:
# --- Define tools using OpenAI's format ---

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": (
                "Get the current weather for a city. "
                "Use this when the user asks about weather conditions."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name (e.g., 'Paris', 'Tokyo')"
                    },
                    "units": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature units",
                    }
                },
                "required": ["city"],
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": (
                "Perform mathematical calculations. "
                "Use this for any math operations the user needs."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Mathematical expression to evaluate (e.g., '2 + 3 * 4')"
                    }
                },
                "required": ["expression"],
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_knowledge_base",
            "description": (
                "Search the company knowledge base for information. "
                "Use this for questions about policies, procedures, products."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query"
                    },
                    "category": {
                        "type": "string",
                        "enum": ["policy", "product", "faq", "general"],
                        "description": "Category to search in"
                    }
                },
                "required": ["query"],
            }
        }
    }
]

print(f"Defined {len(tools)} tools for the LLM")
for t in tools:
    print(f"  🔧 {t['function']['name']}: {t['function']['description'][:60]}...")

### 2.2 The Complete Function Calling Flow

In [ ]:
import json

# --- Tool implementations (the actual code that runs) ---
def get_weather(city: str, units: str = "celsius") -> dict:
    """Simulated weather API."""
    weather_data = {
        "Paris": {"temp": 22, "condition": "Sunny", "humidity": 45},
        "London": {"temp": 15, "condition": "Cloudy", "humidity": 78},
        "Tokyo": {"temp": 28, "condition": "Humid", "humidity": 85},
        "New York": {"temp": 18, "condition": "Windy", "humidity": 52},
    }
    data = weather_data.get(city, {"temp": 20, "condition": "Unknown", "humidity": 50})
    if units == "fahrenheit":
        data["temp"] = round(data["temp"] * 9/5 + 32)
    data["city"] = city
    data["units"] = units
    return data

def calculate(expression: str) -> dict:
    """Safely evaluate math expression."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return {"expression": expression, "result": result}
    except Exception as e:
        return {"expression": expression, "error": str(e)}

def search_knowledge_base(query: str, category: str = "general") -> dict:
    """Simulated knowledge base search."""
    kb = {
        "refund": "Refunds are available within 30 days. Contact support@example.com.",
        "shipping": "Free shipping over $50. Standard delivery: 5-7 business days.",
        "returns": "Returns accepted within 30 days in original packaging.",
        "warranty": "All products come with a 1-year limited warranty.",
    }
    for key, value in kb.items():
        if key in query.lower():
            return {"found": True, "answer": value, "category": category}
    return {"found": False, "answer": "No relevant information found.", "category": category}

# Map tool names to functions
TOOL_MAP = {
    "get_weather": get_weather,
    "calculate": calculate,
    "search_knowledge_base": search_knowledge_base,
}

print("✅ Tool implementations ready!")

In [ ]:
# ============================================================
# THE COMPLETE FUNCTION CALLING FLOW
# ============================================================

# Step 1: Send user message + tools to LLM
print("=" * 60)
print("STEP 1: Send message to LLM with available tools")
print("=" * 60)

messages = [
    {"role": "system", "content": "You are a helpful customer service agent."},
    {"role": "user", "content": "What's the weather in Paris? Also, what's your refund policy?"},
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
    tool_choice="auto",  # Let the LLM decide whether to use tools
)

assistant_message = response.choices[0].message
print(f"Finish reason: {response.choices[0].finish_reason}")
print(f"Tool calls requested: {len(assistant_message.tool_calls or [])}")

# Step 2: Check if the LLM wants to call tools
if assistant_message.tool_calls:
    print(f"
{'=' * 60}")
    print("STEP 2: LLM requested tool calls — executing them")
    print(f"{'=' * 60}")
    
    # Add the assistant's message to history (IMPORTANT: must include tool_calls)
    messages.append(assistant_message)
    
    # Step 3: Execute each tool call
    for tool_call in assistant_message.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        
        print(f"
  🔧 Executing: {function_name}({arguments})")
        
        # Execute the tool
        result = TOOL_MAP[function_name](**arguments)
        result_str = json.dumps(result)
        
        print(f"  📋 Result: {result_str}")
        
        # Step 4: Add tool result to messages
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,  # Must match the tool_call ID
            "content": result_str,
        })
    
    # Step 5: Send everything back to LLM for final response
    print(f"
{'=' * 60}")
    print("STEP 3: Send tool results back to LLM for final answer")
    print(f"{'=' * 60}")
    
    final_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
    )
    
    print(f"
🤖 Agent: {final_response.choices[0].message.content}")
    print(f"
Total tokens used: {final_response.usage.total_tokens}")

else:
    # LLM answered directly without tools
    print(f"
🤖 Agent (no tools needed): {assistant_message.content}")

### 2.3 Tool Choice Options

Control WHEN the LLM should use tools:

In [ ]:
# tool_choice options:

# "auto" — LLM decides whether to call tools (DEFAULT for agents)
# The LLM will call tools when it thinks they're needed
response_auto = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello! How are you?"}],
    tools=tools,
    tool_choice="auto",  # LLM won't call tools for simple greetings
)
print(f"'auto' with greeting → Tool calls: {response_auto.choices[0].message.tool_calls}")
print(f"  Response: {response_auto.choices[0].message.content[:80]}")

# "required" — LLM MUST call at least one tool
response_required = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello!"}],
    tools=tools,
    tool_choice="required",  # Forces tool use even for greetings
)
print(f"
'required' with greeting → Tool calls: {len(response_required.choices[0].message.tool_calls or [])}")

# Specific tool — Force a particular tool
response_specific = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Tell me about Paris"}],
    tools=tools,
    tool_choice={"type": "function", "function": {"name": "get_weather"}},
)
tool_call = response_specific.choices[0].message.tool_calls[0]
print(f"
Forced 'get_weather' → Args: {tool_call.function.arguments}")

# "none" — Disable tool use entirely
response_none = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What's the weather?"}],
    tools=tools,
    tool_choice="none",  # Can't use tools, must answer from knowledge
)
print(f"
'none' → {response_none.choices[0].message.content[:100]}")

### 2.4 Parallel Tool Calls

GPT-4o can call multiple tools in a single response — this is critical for agent efficiency.

In [ ]:
# Ask something that requires multiple tools
messages = [
    {"role": "system", "content": "You are a helpful assistant. Use tools when needed."},
    {"role": "user", "content": (
        "I need three things: "
        "1) Weather in Tokyo, "
        "2) What's 15% tip on $85.50, "
        "3) What's your shipping policy?"
    )},
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
    tool_choice="auto",
)

msg = response.choices[0].message
tool_calls_count = len(msg.tool_calls) if msg.tool_calls else 0
print(f"LLM requested {tool_calls_count} tool calls in ONE response!")
print()

if msg.tool_calls:
    messages.append(msg)
    
    for tc in msg.tool_calls:
        name = tc.function.name
        args = json.loads(tc.function.arguments)
        print(f"  🔧 {name}({args})")
        
        result = TOOL_MAP[name](**args)
        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps(result),
        })
    
    # Get final answer with all tool results
    final = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
    )
    
    print(f"
🤖 Agent:
{final.choices[0].message.content}")

---
# Section 3: Streaming Responses

## Why Streaming Matters for Agents

Without streaming, users stare at a blank screen for 2-10 seconds.
With streaming, they see text appear immediately — much better UX.

```
Without streaming:
  [2-5 second blank wait] → Full response appears at once

With streaming:
  T → Th → The → The w → The weather → The weather in → ... → Complete!
  User sees tokens as they're generated (50-100ms per token)
```

### 3.1 Basic Streaming

In [ ]:
# --- Streaming a response ---
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Explain what an AI agent is in 3 sentences."}
    ],
    stream=True,  # ← This enables streaming
    max_tokens=150,
)

# Each chunk contains a small piece (usually 1 token)
print("Streaming response: ", end="")
full_response = ""
for chunk in stream:
    if chunk.choices[0].delta.content:
        token = chunk.choices[0].delta.content
        full_response += token
        print(token, end="", flush=True)

print(f"

✅ Complete response ({len(full_response)} chars)")

### 3.2 Streaming with Tool Calls

When streaming, tool calls come in chunks too — you need to assemble them:

In [ ]:
import json

# --- Stream a response that includes tool calls ---
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What's the weather in London?"}
    ],
    tools=tools,
    stream=True,
)

# Assemble tool calls from stream chunks
tool_calls_accumulated = {}
content_parts = []
finish_reason = None

for chunk in stream:
    delta = chunk.choices[0].delta
    finish_reason = chunk.choices[0].finish_reason
    
    # Accumulate content
    if delta.content:
        content_parts.append(delta.content)
        print(delta.content, end="", flush=True)
    
    # Accumulate tool calls (they come in pieces)
    if delta.tool_calls:
        for tc in delta.tool_calls:
            idx = tc.index
            if idx not in tool_calls_accumulated:
                tool_calls_accumulated[idx] = {
                    "id": "",
                    "function": {"name": "", "arguments": ""}
                }
            
            if tc.id:
                tool_calls_accumulated[idx]["id"] = tc.id
            if tc.function:
                if tc.function.name:
                    tool_calls_accumulated[idx]["function"]["name"] += tc.function.name
                if tc.function.arguments:
                    tool_calls_accumulated[idx]["function"]["arguments"] += tc.function.arguments

print(f"
Finish reason: {finish_reason}")

if tool_calls_accumulated:
    print(f"
Assembled {len(tool_calls_accumulated)} tool call(s):")
    for idx, tc in tool_calls_accumulated.items():
        print(f"  🔧 {tc['function']['name']}({tc['function']['arguments']})")
else:
    print(f"
Direct response: {''.join(content_parts)}")

### 3.3 Async Streaming (Production Pattern)

In production agents, use async streaming for better performance:

In [ ]:
import asyncio

async def stream_agent_response(user_message: str):
    """
    Production pattern for streaming an agent's response.
    This is how ChatGPT-like interfaces work.
    """
    aclient = openai.AsyncOpenAI()
    
    stream = await aclient.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful AI assistant."},
            {"role": "user", "content": user_message},
        ],
        stream=True,
        max_tokens=200,
    )
    
    full_response = ""
    token_count = 0
    
    print("🤖 Agent: ", end="")
    async for chunk in stream:
        if chunk.choices[0].delta.content:
            token = chunk.choices[0].delta.content
            full_response += token
            token_count += 1
            print(token, end="", flush=True)
    
    print(f"📊 Streamed {token_count} tokens, {len(full_response)} chars")
    return full_response

# Run the async streaming
result = await stream_agent_response("What makes a good AI agent? Be brief.")

---
# Section 4: Anthropic Claude API

## Why Learn Claude?

Claude is often preferred for agents because of:
- **Superior reasoning** — better at complex planning tasks
- **200K context window** — more memory for agents
- **Better instruction following** — less prompt engineering needed
- **Safer outputs** — built-in safety features

### Key Differences from OpenAI

| Feature | OpenAI | Anthropic |
|---------|--------|-----------|
| Auth header | `Authorization: Bearer sk-...` | `x-api-key: sk-ant-...` |
| System prompt | In messages array | Separate `system` parameter |
| Tool format | `tools` parameter | `tools` parameter (similar) |
| Streaming | `stream=True` | `stream=True` |
| Max tokens | Optional | **Required** |
| Vision | In messages | In messages (similar) |

### 4.1 Basic Claude API Call

In [ ]:
import anthropic

# Initialize Anthropic client
claude_client = anthropic.Anthropic()

# --- Basic message ---
response = claude_client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,                    # Required for Claude!
    system="You are a helpful AI assistant. Be concise.",  # Separate system param
    messages=[
        {"role": "user", "content": "What is an AI agent in 2 sentences?"}
    ],
)

print(f"Model: {response.model}")
print(f"Content: {response.content[0].text}")
print(f"Stop reason: {response.stop_reason}")
print(f"\nTokens — Input: {response.usage.input_tokens}, Output: {response.usage.output_tokens}")

### 4.2 Claude Tool Use (Function Calling)

In [ ]:
# --- Claude's tool format ---
claude_tools = [
    {
        "name": "get_weather",
        "description": "Get current weather for a city.",
        "input_schema": {  # Note: "input_schema" not "parameters"
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "City name"
                }
            },
            "required": ["city"]
        }
    },
    {
        "name": "calculate", 
        "description": "Evaluate a math expression.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "Math expression"
                }
            },
            "required": ["expression"]
        }
    }
]

# --- Send with tools ---
response = claude_client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=500,
    system="You are a helpful assistant. Use tools when needed.",
    messages=[
        {"role": "user", "content": "What's the weather in Tokyo and what's 25 * 17?"}
    ],
    tools=claude_tools,
)

print(f"Stop reason: {response.stop_reason}")
print(f"Content blocks: {len(response.content)}")

# Process response — Claude can return text AND tool_use blocks
tool_results = []
for block in response.content:
    if block.type == "text":
        print(f"\nText: {block.text}")
    elif block.type == "tool_use":
        print(f"\n🔧 Tool call: {block.name}({json.dumps(block.input)})")
        print(f"   Tool use ID: {block.id}")
        
        # Execute the tool
        result = TOOL_MAP[block.name](**block.input)
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": json.dumps(result),
        })

# Send results back to Claude
if tool_results and response.stop_reason == "tool_use":
    final_response = claude_client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        system="You are a helpful assistant.",
        messages=[
            {"role": "user", "content": "What's the weather in Tokyo and what's 25 * 17?"},
            {"role": "assistant", "content": response.content},
            {"role": "user", "content": tool_results},  # Tool results go as user message
        ],
        tools=claude_tools,
    )
    print(f"\n🤖 Final: {final_response.content[0].text}")

---
# Section 5: Prompt Engineering for Agents

## Why Agent Prompts Are Different

Agent prompts are much more complex than chatbot prompts because they must:
1. Define the agent's **identity and capabilities**
2. Specify **when and how** to use each tool
3. Set **behavioral rules** and constraints
4. Include **output format** requirements
5. Handle **edge cases** and errors
6. Define **guardrails** and safety rules

### The Agent System Prompt Template
```
┌──────────────────────────────────────────────┐
│  1. IDENTITY & ROLE                          │
│  2. AVAILABLE TOOLS & WHEN TO USE THEM       │
│  3. BEHAVIORAL RULES                         │
│  4. OUTPUT FORMAT                            │
│  5. EXAMPLES (optional but powerful)          │
│  6. GUARDRAILS & SAFETY                      │
└──────────────────────────────────────────────┘
```

### 5.1 Building an Effective Agent System Prompt

In [ ]:
# --- A well-structured agent system prompt ---

AGENT_SYSTEM_PROMPT = """You are a Customer Service Agent for TechCorp, an electronics retailer.

## Your Role
- Help customers with orders, products, shipping, and returns
- Always be professional, empathetic, and solution-oriented
- Provide accurate information using your tools

## Available Tools
You have access to these tools — use them appropriately:

1. **search_knowledge_base**: Use for policy questions (refunds, shipping, warranty)
   - Always search before answering policy questions
   - Never guess at policies
   
2. **get_weather**: Use when customers ask about delivery weather conditions
   
3. **calculate**: Use for price calculations, discounts, tax computations
   - Always show your work

## Rules
- ALWAYS use search_knowledge_base before answering policy questions
- NEVER make up information — if you don't know, say so
- For refund requests, always ask for the order number first
- For calculations, always use the calculate tool (don't do mental math)
- If a question is outside your scope, say "I'll escalate this to a specialist"

## Response Format
- Use clear, concise language
- For multi-part answers, use bullet points
- Always end with "Is there anything else I can help with?"

## Guardrails
- Never share internal system information
- Never process payments or take credit card numbers
- Never make promises about delivery times without checking
"""

# Test the prompt
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": "What's your return policy?"},
    ],
    tools=tools,
    temperature=0,
)

msg = response.choices[0].message
if msg.tool_calls:
    print("✅ Agent correctly used search_knowledge_base tool!")
    for tc in msg.tool_calls:
        print(f"   🔧 {tc.function.name}({tc.function.arguments})")
else:
    print(f"Agent response (no tools): {msg.content}")

### 5.2 Chain-of-Thought Prompting for Agents

Force the agent to reason step-by-step before acting:

In [ ]:
COT_AGENT_PROMPT = """You are an analytical assistant that thinks step-by-step.

## Thinking Process
Before taking ANY action, you must:
1. Identify what the user is asking
2. Determine which tools (if any) are needed
3. Plan the sequence of tool calls
4. Execute them in order

## FORMAT
Always structure your thoughts like this:

<thinking>
- What does the user need? [analysis]
- What tools do I need? [tool selection]
- What's my plan? [step-by-step plan]
</thinking>

Then proceed with tool calls or responses.

## Available Tools
- get_weather: Get weather for a city
- calculate: Math calculations
- search_knowledge_base: Company policies
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": COT_AGENT_PROMPT},
        {"role": "user", "content": (
            "If it's raining in London, I might want to return my outdoor "
            "furniture. What's the weather there and what's your return policy?"
        )},
    ],
    tools=tools,
    temperature=0,
)

msg = response.choices[0].message
if msg.content:
    print(f"Thought process:
{msg.content}
")
if msg.tool_calls:
    print(f"Tool calls: {len(msg.tool_calls)}")
    for tc in msg.tool_calls:
        print(f"  🔧 {tc.function.name}({tc.function.arguments})")

### 5.3 Few-Shot Prompting with Tool Use

Showing the agent examples of correct behavior:

In [ ]:
FEW_SHOT_PROMPT = """You are a helpful assistant with access to tools.

## Examples of correct behavior:

### Example 1: User asks about weather
User: "What's it like in Paris today?"
Your reasoning: The user wants weather info. I should use get_weather.
Action: Call get_weather(city="Paris")

### Example 2: User asks a math question  
User: "If I buy 3 items at $29.99 each with 8.5% tax, what's the total?"
Your reasoning: This requires calculation. I'll use the calculate tool.
Action: Call calculate(expression="3 * 29.99 * 1.085")

### Example 3: User asks a policy question
User: "Can I return something?"
Your reasoning: This is a policy question. I must search the knowledge base first.
Action: Call search_knowledge_base(query="returns", category="policy")

### Example 4: User makes conversation (no tool needed)
User: "Thanks for your help!"
Your reasoning: This is just conversation. No tool needed.
Action: Respond directly with "You're welcome! Is there anything else?"

Now handle the actual user query:
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": FEW_SHOT_PROMPT},
        {"role": "user", "content": "What would 20% off $149.99 be?"},
    ],
    tools=tools,
    temperature=0,
)

msg = response.choices[0].message
if msg.tool_calls:
    tc = msg.tool_calls[0]
    print(f"✅ Correctly called: {tc.function.name}({tc.function.arguments})")
if msg.content:
    print(f"Response: {msg.content}")

---
# Section 6: Structured Outputs — Reliable JSON from LLMs

## Why Structured Outputs Matter

When building agents, you often need the LLM to return data in a **specific format**:
- Tool call decisions (which tool, what arguments)
- Agent state updates (status, next steps)
- API responses (formatted data)

Without structured outputs, you're parsing free text — fragile and error-prone.
With structured outputs, you get **guaranteed valid JSON**.

### 6.1 JSON Mode (Simple)

In [ ]:
# --- JSON Mode: Force the LLM to output valid JSON ---
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "You are an AI that analyzes user intent. "
                "Output JSON with: intent, entities, confidence, needs_tools (bool)"
            )
        },
        {"role": "user", "content": "What's the weather like in Tokyo?"}
    ],
    response_format={"type": "json_object"},  # ← Force JSON output
    temperature=0,
)

result = json.loads(response.choices[0].message.content)
print("Parsed JSON output:")
print(json.dumps(result, indent=2))
print(f"Type: {type(result)} — Guaranteed valid JSON!")

### 6.2 JSON Schema Mode (Strict — Recommended for Agents)

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional

# --- Define the expected output structure with Pydantic ---
class AgentDecision(BaseModel):
    """The agent's decision about what to do next."""
    reasoning: str = Field(description="Step-by-step reasoning")
    action: Literal["use_tool", "respond_directly", "ask_clarification", "escalate"] = Field(
        description="What action to take"
    )
    tool_name: Optional[str] = Field(default=None, description="Tool to call if action is use_tool")
    tool_args: Optional[dict] = Field(default=None, description="Arguments for the tool")
    response: Optional[str] = Field(default=None, description="Direct response if not using tool")
    confidence: float = Field(description="Confidence 0-1", ge=0, le=1)

# --- Use Pydantic with OpenAI's structured output ---
response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You are an AI agent. Analyze the user's request and decide what to do."
        },
        {"role": "user", "content": "How much is 15% tip on $84.50?"}
    ],
    response_format=AgentDecision,
)

decision = response.choices[0].message.parsed
print(f"Reasoning: {decision.reasoning}")
print(f"Action: {decision.action}")
print(f"Tool: {decision.tool_name}")
print(f"Tool args: {decision.tool_args}")
print(f"Confidence: {decision.confidence}")
print(f"
✅ Type-safe, validated output — no parsing errors possible!")

In [ ]:
# --- Test with different queries ---
test_queries = [
    "What's the weather in Paris?",
    "Thanks for your help!",
    "Can I speak to a manager?",
    "I'm not sure what I need...",
]

for query in test_queries:
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are an AI agent. Decide what to do."},
            {"role": "user", "content": query}
        ],
        response_format=AgentDecision,
    )
    d = response.choices[0].message.parsed
    print(f"Query: '{query}'")
    print(f"  → Action: {d.action} | Tool: {d.tool_name} | Confidence: {d.confidence}")
    print()

---
# Section 7: Token Management & Cost Control

## Why This Matters in Production

LLM API costs scale with token usage. A single agent run can use 5K-50K tokens.
At scale (1000s of requests/day), costs can explode without management.

### Token Pricing (approximate, 2025-2026):

| Model | Input (/1M tokens) | Output (/1M tokens) | Best For |
|-------|-------------------|---------------------|----------|
| `gpt-4o-mini` | $0.15 | $0.60 | High-volume, simple tasks |
| `gpt-4o` | $2.50 | $10.00 | Complex reasoning |
| `o3-mini` | $1.10 | $4.40 | Reasoning tasks |
| `claude-3.5-haiku` | $0.80 | $4.00 | Fast, cheap Claude |
| `claude-sonnet-4` | $3.00 | $15.00 | Best Claude reasoning |

### 7.1 Counting Tokens with `tiktoken`

In [4]:
import tiktoken

# Get the tokenizer for a specific model
enc = tiktoken.encoding_for_model("gpt-4o")

# --- Count tokens in text ---
text = "AI agents are autonomous systems that use LLMs to reason and take actions 1997."
tokens = enc.encode(text)
print(f"Text: '{text}'")
print(f"Token count: {len(tokens)}")
print(f"Tokens: {tokens[:10]}...")  # Show first 10 token IDs

# --- See what each token looks like ---
print(f"Token breakdown:")
for token_id in tokens:
    decoded = enc.decode([token_id])
    print(f"  {token_id:>6} → '{decoded}'")

Text: 'AI agents are autonomous systems that use LLMs to reason and take actions 1997.'
Token count: 19
Tokens: [17527, 19297, 553, 78677, 7511, 484, 1199, 451, 19641, 82]...
Token breakdown:
   17527 → 'AI'
   19297 → ' agents'
     553 → ' are'
   78677 → ' autonomous'
    7511 → ' systems'
     484 → ' that'
    1199 → ' use'
     451 → ' L'
   19641 → 'LM'
      82 → 's'
     316 → ' to'
    5207 → ' reason'
     326 → ' and'
    2304 → ' take'
   10370 → ' actions'
     220 → ' '
    3204 → '199'
      22 → '7'
      13 → '.'


In [ ]:
# --- Count tokens in a full message array (like an agent's context) ---

def count_message_tokens(messages: list[dict], model: str = "gpt-4o") -> dict:
    """
    Count tokens in a message array.
    This is crucial for staying within context window limits.
    """
    enc = tiktoken.encoding_for_model(model)
    
    total_tokens = 0
    breakdown = []
    
    for msg in messages:
        # Each message has overhead tokens (role, separators)
        msg_tokens = 4  # ~4 tokens per message for formatting
        
        content = msg.get("content", "")
        if isinstance(content, str):
            content_tokens = len(enc.encode(content))
        else:
            content_tokens = len(enc.encode(str(content)))
        
        msg_tokens += content_tokens
        msg_tokens += len(enc.encode(msg.get("role", "")))
        
        total_tokens += msg_tokens
        breakdown.append({
            "role": msg["role"],
            "content_preview": content[:50] + "..." if len(content) > 50 else content,
            "tokens": msg_tokens,
        })
    
    total_tokens += 3  # Reply priming tokens
    
    return {
        "total_tokens": total_tokens,
        "breakdown": breakdown,
        "estimated_cost_gpt4o": total_tokens / 1_000_000 * 2.50,
        "estimated_cost_gpt4o_mini": total_tokens / 1_000_000 * 0.15,
    }

# Test with a sample agent conversation
messages = [
    {"role": "system", "content": AGENT_SYSTEM_PROMPT},
    {"role": "user", "content": "What's your return policy and how much is 20% off $149?"},
    {"role": "assistant", "content": "Let me check that for you. I'll look up our return policy and calculate the discount."},
    {"role": "user", "content": "Also, what's the weather in New York?"},
]

stats = count_message_tokens(messages)
print("Token Usage Analysis:")
print(f"  Total tokens: {stats['total_tokens']}")
print(f"  Est. cost (GPT-4o): ${stats['estimated_cost_gpt4o']:.6f}")
print(f"  Est. cost (GPT-4o-mini): ${stats['estimated_cost_gpt4o_mini']:.6f}")
print(f"
Breakdown:")
for b in stats['breakdown']:
    print(f"  [{b['role']:>10}] {b['tokens']:>4} tokens — {b['content_preview']}")

### 7.2 Cost Tracking for Production Agents

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime

@dataclass
class CostTracker:
    """Track LLM API costs across agent runs."""
    
    # Pricing per 1M tokens (update as prices change)
    PRICING = {
        "gpt-4o": {"input": 2.50, "output": 10.00},
        "gpt-4o-mini": {"input": 0.15, "output": 0.60},
        "o3-mini": {"input": 1.10, "output": 4.40},
        "claude-sonnet-4-20250514": {"input": 3.00, "output": 15.00},
        "claude-3-5-haiku-20241022": {"input": 0.80, "output": 4.00},
    }
    
    total_input_tokens: int = 0
    total_output_tokens: int = 0
    total_cost: float = 0.0
    calls: list = field(default_factory=list)
    budget_limit: float = 1.0  # $1 default budget
    
    def record_call(self, model: str, input_tokens: int, output_tokens: int):
        """Record an API call and its cost."""
        pricing = self.PRICING.get(model, {"input": 5.0, "output": 15.0})
        
        input_cost = (input_tokens / 1_000_000) * pricing["input"]
        output_cost = (output_tokens / 1_000_000) * pricing["output"]
        call_cost = input_cost + output_cost
        
        self.total_input_tokens += input_tokens
        self.total_output_tokens += output_tokens
        self.total_cost += call_cost
        
        self.calls.append({
            "timestamp": datetime.now().isoformat(),
            "model": model,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "cost": call_cost,
        })
        
        return call_cost
    
    def check_budget(self) -> bool:
        """Check if we're within budget."""
        return self.total_cost < self.budget_limit
    
    def summary(self) -> str:
        """Get cost summary."""
        return (
            f"API Calls: {len(self.calls)} | "
            f"Tokens: {self.total_input_tokens:,} in + {self.total_output_tokens:,} out | "
            f"Cost: ${self.total_cost:.4f} / ${self.budget_limit:.2f} budget"
        )

# --- Demo ---
tracker = CostTracker(budget_limit=0.10)  # 10 cent budget

# Simulate some agent API calls
tracker.record_call("gpt-4o-mini", 500, 150)
tracker.record_call("gpt-4o-mini", 1200, 300)
tracker.record_call("gpt-4o", 800, 200)  # More expensive call
tracker.record_call("gpt-4o-mini", 600, 100)

print("Cost Tracking Summary:")
print(tracker.summary())
print(f"Within budget: {'✅' if tracker.check_budget() else '❌'}")
print(f"
Per-call breakdown:")
for call in tracker.calls:
    print(f"  {call['model']:>20}: {call['input_tokens']:>5} in + {call['output_tokens']:>4} out = ${call['cost']:.6f}")

---
# Section 8: Capstone — Build an LLM Router

## Smart Model Selection for Cost Optimization

In production, using GPT-4o for everything is **expensive**.
An LLM Router analyzes the query and picks the cheapest model that can handle it.

```
User Query → Router → Simple? → GPT-4o-mini ($0.15/M)
                    → Complex? → GPT-4o ($2.50/M)
                    → Reasoning? → o3-mini ($1.10/M)

Savings: 60-80% by routing simple queries to cheaper models
```

In [ ]:
import json
from pydantic import BaseModel, Field
from typing import Literal

class RouteDecision(BaseModel):
    """Router's decision about which model to use."""
    complexity: Literal["simple", "moderate", "complex"] = Field(
        description="Query complexity level"
    )
    needs_reasoning: bool = Field(description="Whether deep reasoning is needed")
    needs_tools: bool = Field(description="Whether tool use is likely needed")
    recommended_model: str = Field(description="Recommended model")
    reasoning: str = Field(description="Why this model was chosen")

class LLMRouter:
    """
    Routes queries to the most cost-effective model.
    
    Uses a cheap model (gpt-4o-mini) to classify queries,
    then routes to the appropriate model. This saves money by
    only using expensive models when needed.
    """
    
    MODEL_MAP = {
        "simple": "gpt-4o-mini",       # $0.15/M in  — Greetings, simple Q&A
        "moderate": "gpt-4o-mini",      # $0.15/M in  — Most tool-calling tasks
        "complex": "gpt-4o",            # $2.50/M in  — Complex reasoning
        "reasoning": "o3-mini",         # $1.10/M in  — Math, logic, planning
    }
    
    ROUTER_PROMPT = """Classify the user query complexity.
    
Rules:
- simple: Greetings, thanks, simple factual questions
- moderate: Questions needing 1-2 tool calls, straightforward analysis
- complex: Multi-step reasoning, complex analysis, planning, coding
    
Also determine if deep reasoning or tool use is needed."""
    
    def __init__(self):
        self.client = openai.OpenAI()
        self.route_log = []
    
    def route(self, user_query: str) -> dict:
        """Analyze query and return the best model to use."""
        
        # Use cheap model to classify
        response = self.client.beta.chat.completions.parse(
            model="gpt-4o-mini",  # Always use cheapest for routing
            messages=[
                {"role": "system", "content": self.ROUTER_PROMPT},
                {"role": "user", "content": user_query},
            ],
            response_format=RouteDecision,
            temperature=0,
        )
        
        decision = response.choices[0].message.parsed
        
        # Determine model
        if decision.needs_reasoning:
            model = self.MODEL_MAP["reasoning"]
        else:
            model = self.MODEL_MAP[decision.complexity]
        
        result = {
            "model": model,
            "complexity": decision.complexity,
            "needs_tools": decision.needs_tools,
            "reasoning": decision.reasoning,
            "routing_tokens": response.usage.total_tokens,
        }
        
        self.route_log.append(result)
        return result

# --- Test the router ---
router = LLMRouter()

test_queries = [
    "Hi there!",
    "What's the weather in Tokyo?",
    "Analyze the pros and cons of 5 different database architectures for a high-throughput real-time system with complex queries",
    "What's 15% of $84.50?",
    "Write a Python function to implement a red-black tree with insertion, deletion, and rebalancing",
]

print("LLM Router Results:")
print("=" * 70)
for query in test_queries:
    result = router.route(query)
    print(f"
Query: '{query[:60]}...'")
    print(f"  → Model: {result['model']} ({result['complexity']})")
    print(f"  → Needs tools: {result['needs_tools']}")
    print(f"  → Reason: {result['reasoning'][:80]}...")

---
# ✅ Week 2 Complete!

## What You Learned

| Skill | What You Can Do Now |
|-------|---------------------|
| **OpenAI API** | Chat completions, model selection, parameters |
| **Function Calling** | Define tools, handle tool calls, parallel execution |
| **Streaming** | Stream responses and tool calls in real-time |
| **Anthropic API** | Claude messages, tool use, key differences |
| **Prompt Engineering** | System prompts, CoT, few-shot for agents |
| **Structured Outputs** | JSON mode, Pydantic parsing, guaranteed schemas |
| **Token Management** | Count tokens, track costs, set budgets |
| **LLM Router** | Smart model selection for cost optimization |

## Next Week Preview: Week 3 — Building Your First Agent

- The complete agent loop from scratch
- ReAct pattern implementation
- Memory management
- Multi-turn tool calling
- Error recovery
- Putting it ALL together into a working agent

---
*Part of the [AI Agents from Scratch Guide](AI-Agents-From-Scratch-Guide.md)*